In [0]:
import pyspark.sql.functions as F

# ---------------- CONFIG ----------------
CATALOG = "chatbot_dev"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

# Ensure gold schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

# Load
patient_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.patient_data").drop("source_file").drop("ingested_at")
appointment_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.appointments_data").drop("source_file").drop("ingested_at")
lab_result_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.lab_results_data").drop("source_file").drop("ingested_at")
symptom_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.symptoms_data").drop("source_file").drop("ingested_at")
medical_history_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.medical_history").drop("source_file").drop("ingested_at")
medication_df=spark.read.table(f"{CATALOG}.{SILVER_SCHEMA}.medication_data").drop("source_file").drop("ingested_at")
dim_table = f"{CATALOG}.{GOLD_SCHEMA}.dim_patient"

# Join (LEFT JOIN so you keep all patients)
dim = (
    patient_df.alias("a")
    .join(appointment_df.alias("b"), "patient_id", "left")
    .select(
        "a.*",
        col("b.doctor_name").alias("appointment_doctor_name"),
        col("b.appointment_date").alias("appointment_date"),
        col("b.reason_for_visit").alias("reason_for_visit"),
        col("b.status").alias("status")
    )
    .join(lab_result_df.alias("c"), "patient_id", "left")
    .join(symptom_df.alias("d"), "patient_id", "left")
    .join(medical_history_df.alias("e"), "patient_id", "left")
    .join(medication_df.alias("f"), "patient_id", "left")
)

# creating new column with merged text from all the columns, later will use this column to generate embeddings
dim_merged = dim.withColumn(
    "combined_text",
    F.concat_ws(
        " | ",
        *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in dim.columns]
    )
)

# Write one dimension table
(dim_merged.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(dim_table)
)

In [0]:
%sql
select * from chatbot_dev.gold.dim_patient